# TACS Adaptive Top-K + Diverse Context - SUN397

Bien the nay phat trien tu notebook TACS top-3: selector cham diem toan candidate pool 20%, chon context bang MMR de can bang task utility va diversity. So context duoc chon thay doi tu 1 den 3.

- Train random K=1..3 theo batch; Gumbel-ST va policy reward su dung cung diversity-aware score.
- Validation/test dung MMR tao danh sach toi da 3 context; nguong adaptive duoc chon tren validation.
- Giu SUN397 split, DINOv3, augmentation va cac loss/optimizer cua baseline.
- Khong dung captions.csv; context la anh trong candidate pool.
- Day la bien the thuc nghiem, khong phai TACS nguyen ban. Checkpoint/output duoc tach rieng.


In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
# !unzip -q /content/drive/MyDrive/archive\ \(13\).zip -d /content/archive\ \(13\)/

## 1. Cài thư viện

Chạy cell này trên Colab nếu môi trường chưa có đủ thư viện. Nếu chạy local đã cài sẵn thì có thể bỏ qua.


In [3]:
%pip -q install torch torchvision pandas pillow tqdm matplotlib torchmetrics


Note: you may need to restart the kernel to use updated packages.


In [4]:
# !wget -c 'https://dinov3.llamameta.net/dinov3_vits16/dinov3_vits16_pretrain_lvd1689m-08c60483.pth?Policy=eyJTdGF0ZW1lbnQiOlt7InVuaXF1ZV9oYXNoIjoiNTFma3Jxa3l3bTl3MGpweHI2NndvNWZsIiwiUmVzb3VyY2UiOiJodHRwczpcL1wvZGlub3YzLmxsYW1hbWV0YS5uZXRcLyoiLCJDb25kaXRpb24iOnsiRGF0ZUxlc3NUaGFuIjp7IkFXUzpFcG9jaFRpbWUiOjE3ODkyOTc0MTJ9fX1dfQ__&Signature=sdowk%7EiiF6j8599sB9K5-M3sB%7Elq1be-oKP%7El7VBISEmjHEBPyCWw9Y2JD4oJY9LbXE313Q4ZgjX%7EXO3EiwGQZ3MZhfkZr0OXmADX8L%7E7XX%7E4mkVA1JSOkVzEa3Oj39gY%7E%7ER21%7EmJXEcFOZPMj%7EzGs-uxIh1xeRrfo-ZUqv3Yz7z7vUgn1OoDhu98kPnh8KT4MF1Gd6gcB4LogqWk3ooVScR6z8ECw50eawfY8lyUUzv%7E%7Em1QdstSU1k6HFhKFbuuogf0MyH7rzYScskxtGG95ke47CKn6CWSNbHLKhcLDfNx-XrN0A-xxis10rpJaMh2jBmxwohSr8XQlGVCBPWqQ__&Key-Pair-Id=K15QRJLYKIFSLZ&Download-Request-ID=1749739516259745' \
#   -O /content/dinov3_vits16_pretrain_lvd1689m-08c60483.pth

## 2. Import, cấu hình, dataset và candidate pool

Cell này đọc standard SUN397 split nếu có `Training_01.txt` / `Testing_01.txt`; nếu không có thì tự tạo split 50 train / 50 test mỗi class. Candidate pool lấy **20% train split** như paper.

Bản này không dùng random shortlist nữa. Trong train/eval, mỗi query được tính utility score với toàn bộ candidate pool.


In [5]:
import json
import random
import subprocess
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm

torch.backends.cudnn.benchmark = True
# Tận dụng Tensor Cores thế hệ mới trên dòng RTX / GPU Ampere+
torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:', DEVICE)

# --- ĐƯỜNG DẪN OUTPUT (Ghi vào working của Kaggle) ---
PROJECT_DIR = Path('/kaggle/working')
OUT_PUT_DIR = PROJECT_DIR / 'outputs' / 'tacs_adaptive_diverse_sun_true_val_split'
OUTPUT_DIR = OUT_PUT_DIR
OUT_PUT_DIR.mkdir(parents=True, exist_ok=True)

# --- ĐƯỜNG DẪN DATASET (SUN397) ---
DATASET_ROOT = Path('/kaggle/input/datasets/hiuphmnhtrung/sun397/SUN397/SUN397')
if not DATASET_ROOT.exists():
    fallback_candidates = [
        Path('/kaggle/input/sun397/SUN397/SUN397'),
        Path('/kaggle/input/sun397/SUN397'),
        Path('/kaggle/input/sun397'),
        Path('/kaggle/input/datasets/hiuphmnhtrung/sun397/SUN397'),
    ]
    for candidate in fallback_candidates:
        if candidate.exists():
            DATASET_ROOT = candidate
            break

IMAGE_ROOT = DATASET_ROOT

# --- ĐƯỜNG DẪN DINOv3 REPO & WEIGHTS ---
DINOV3_REPO_DIR = '/kaggle/input/datasets/sinhnguync/dinov3-code/dinov3-main'
if not Path(DINOV3_REPO_DIR).exists():
    for p in [Path('/kaggle/input/dinov3-code/dinov3-main'), Path('/kaggle/input/dinov3-code')]:
        if p.exists():
            DINOV3_REPO_DIR = str(p)
            break

DINOV3_MODEL_NAME = 'dinov3_vits16'
DINOV3_WEIGHTS = '/kaggle/input/datasets/sinhnguync/dinov3/dinov3_vits16_pretrain_lvd1689m-08c60483.pth'
if not Path(DINOV3_WEIGHTS).exists():
    for p in [
        Path('/kaggle/input/dinov3/dinov3_vits16_pretrain_lvd1689m-08c60483.pth'),
        Path('/kaggle/input/dinov3/dinov3_vits16.pth'),
    ]:
        if p.exists():
            DINOV3_WEIGHTS = str(p)
            break

BACKBONE_NAME = DINOV3_MODEL_NAME
FEATURE_DIM = 384

IMAGE_SIZE = 384
BATCH_SIZE = 64 # K nho ngau nhien 1..3 giup gioi han VRAM
EPOCHS = 100
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
BACKBONE_LR = 3e-6
HEAD_LR = 1e-4
BACKBONE_WEIGHT_DECAY = 0.05
HEAD_WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
TOKEN_HEAD_DROPOUT = 0.3
FREEZE_BACKBONE_EPOCHS = 0
WARMUP_EPOCHS = 5
PATIENCE = 10
GRAD_CLIP_NORM = 1.0
NUM_WORKERS = 0
NUM_CLASSES = 397  # SUN397 có 397 scene categories
TOP_K = 5
CONTEXT_TOP_K = 3  # So context toi da
DIVERSITY_WEIGHT = 0.35
ADAPTIVE_SCORE_GAP_THRESHOLD = 0.75
ADAPTIVE_THRESHOLD_GRID = [0.0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0]
RETRIEVAL_SCORE_WEIGHT = 0.0
SUPCON_WEIGHT = 0.05
SUPCON_TEMPERATURE = 0.10
USE_BBOX_CROP = False  # SUN397 là bài toán scene classification, không dùng crop bbox
CANDIDATE_POOL_FRACTION = 0.20
VAL_FRACTION = 0.10
RETRIEVAL_BATCH_SIZE = 512
RETRIEVAL_CHUNK_SIZE = 4096
QUERY_CONTEXT_GATE_DROPOUT = 0.20
QUERY_CONTEXT_GATE_BIAS = -0.7
EVAL_TTA_HORIZONTAL_FLIP = False
USE_CONTEXT_LABEL_PRIOR = False
CONTEXT_LABEL_PRIOR_WEIGHT = 0.25
CONTEXT_LABEL_PRIOR_TEMPERATURE = 0.07

TRAIN_ON_FULL_TRAIN = False
FULL_TRAIN_MONITOR_FRACTION = 0.05  # chi dung khi TRAIN_ON_FULL_TRAIN=True

# --- TACS selector hyperparameters ---
TACS_CANDIDATES_PER_QUERY = None  # None nghĩa là dùng toàn bộ candidate pool
TACS_PREFILTER_TOP_SIMILAR = 0
TACS_GUMBEL_TAU = 0.10
TACS_POLICY_WEIGHT = 1.0
TACS_ENTROPY_WEIGHT = 0.01
TACS_SELECTOR_LR = 1e-4
TACS_SELECTOR_WEIGHT_DECAY = 1e-4
TACS_SELECTOR_DROPOUT = 0.10
TACS_SELECTOR_TEMPERATURE = 1.0

# --- Full-pool selector memory controls ---
TACS_USE_FULL_CANDIDATE_POOL = True
CANDIDATE_IMAGE_BANK_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CANDIDATE_IMAGE_BANK_DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32
CANDIDATE_FEATURE_BANK_BATCH_SIZE = 128
FULL_POOL_SCORE_CHUNK_SIZE = 1024

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE + 48, IMAGE_SIZE + 48)),
    transforms.RandomResizedCrop(
        (IMAGE_SIZE, IMAGE_SIZE),
        scale=(0.60, 1.0),
        ratio=(0.80, 1.25),
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([
        transforms.ColorJitter(
            brightness=0.25,
            contrast=0.25,
            saturation=0.20,
            hue=0.04,
        )
    ], p=0.75),
    transforms.RandomGrayscale(p=0.05),
    transforms.RandomApply([
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0))
    ], p=0.15),
    transforms.ToTensor(),
    transforms.RandomErasing(
        p=0.20,
        scale=(0.02, 0.12),
        ratio=(0.3, 3.3),
        value="random",
    ),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

_mean = globals().get("DINO_MEAN", globals().get("IMAGENET_MEAN", (0.485, 0.456, 0.406)))
_std = globals().get("DINO_STD", globals().get("IMAGENET_STD", (0.229, 0.224, 0.225)))

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=_mean, std=_std),
])


def load_class_names(dataset_root):
    classes_path = dataset_root / 'ClassName.txt'
    if not classes_path.is_file():
        fallback_paths = [
            dataset_root.parent / 'ClassName.txt',
            dataset_root / 'classes.txt',
            Path('/kaggle/input/datasets/hiuphmnhtrung/sun397/SUN397/SUN397/ClassName.txt'),
            Path('/kaggle/input/sun397/SUN397/SUN397/ClassName.txt'),
        ]
        for p in fallback_paths:
            if p.is_file():
                classes_path = p
                break

    if not classes_path.is_file():
        raise FileNotFoundError(f"Không tìm thấy ClassName.txt tại: {classes_path}")

    class_names = []
    with open(classes_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                # Dòng có dạng /a/abbey hoặc a/abbey -> chuẩn hóa bỏ dấu / ở đầu
                class_names.append(line.lstrip('/'))

    global NUM_CLASSES
    if len(class_names) != NUM_CLASSES:
        print(f"Cập nhật NUM_CLASSES từ {NUM_CLASSES} thành {len(class_names)}")
        NUM_CLASSES = len(class_names)
    return class_names


def load_sun397_annotations(dataset_root, image_root, class_names):
    class_to_id = {cls.lstrip('/'): idx for idx, cls in enumerate(class_names)}

    # 1. Tìm các file partition chuẩn (Training_01.txt / Testing_01.txt) nếu có sẵn trong dataset
    train_part_file, test_part_file = None, None
    search_dirs = [
        dataset_root,
        dataset_root.parent / 'sun397_other',
        dataset_root.parent / 'sun397_other' / 'Partitions',
        dataset_root.parent.parent / 'sun397_other',
        Path('/kaggle/input/datasets/hiuphmnhtrung/sun397/SUN397/sun397_other'),
    ]
    for d in search_dirs:
        if d.is_dir():
            t1 = d / 'Training_01.txt'
            te1 = d / 'Testing_01.txt'
            if t1.is_file() and te1.is_file():
                train_part_file, test_part_file = t1, te1
                break
            found_t = list(d.glob('**/Training_01.txt'))
            if found_t:
                train_part_file = found_t[0]
                cand_te = train_part_file.parent / 'Testing_01.txt'
                if cand_te.is_file():
                    test_part_file = cand_te
                    break

    records = []
    if train_part_file and test_part_file:
        print(f"Sử dụng official partition: {train_part_file.name} và {test_part_file.name}")
        for split_label, p_file in [('train', train_part_file), ('test', test_part_file)]:
            with open(p_file, 'r', encoding='utf-8') as f:
                for line in f:
                    rel_p = line.strip().lstrip('/')
                    if not rel_p:
                        continue
                    class_folder = str(Path(rel_p).parent).replace('\\', '/').lstrip('/')
                    if class_folder not in class_to_id:
                        continue
                    class_id = class_to_id[class_folder]
                    full_p = image_root / rel_p
                    if not full_p.is_file():
                        if (dataset_root / rel_p).is_file():
                            full_p = dataset_root / rel_p
                        else:
                            continue
                    records.append({
                        'rel_path': rel_p,
                        'fname': Path(rel_p).name,
                        'image_path': str(full_p.resolve()),
                        'class_id': int(class_id),
                        'is_train': 1 if split_label == 'train' else 0,
                    })
        df = pd.DataFrame(records)
    else:
        print("Không thấy file partition, tự động quét thư mục ảnh theo ClassName.txt và chia split 50/50 chuẩn...")
        all_rows = []
        rng = np.random.default_rng(SEED)
        valid_exts = {'.jpg', '.jpeg', '.png', '.bmp'}

        for cls_name in tqdm(class_names, desc="Đang đọc các lớp SUN397"):
            cls_dir = image_root / cls_name
            if not cls_dir.is_dir():
                cls_dir = dataset_root / cls_name
            if not cls_dir.is_dir():
                continue

            cls_id = class_to_id[cls_name]
            img_paths = [p for p in cls_dir.iterdir() if p.is_file() and p.suffix.lower() in valid_exts]
            img_paths.sort()
            if not img_paths:
                continue

            n_total = len(img_paths)
            indices = np.arange(n_total)
            rng.shuffle(indices)

            # Benchmark chuẩn SUN397: 50 train / 50 test mỗi class nếu >= 100 ảnh, hoặc chia đôi
            if n_total >= 100:
                n_train = 50
                n_test = 50
                train_idx = set(indices[:n_train])
                test_idx = set(indices[n_train:n_train + n_test])
            else:
                n_train = max(1, n_total // 2)
                train_idx = set(indices[:n_train])
                test_idx = set(indices[n_train:])

            for idx, p in enumerate(img_paths):
                if idx in train_idx:
                    is_train = 1
                elif idx in test_idx:
                    is_train = 0
                else:
                    continue

                try:
                    rel_p = str(p.relative_to(image_root)).replace('\\', '/')
                except Exception:
                    rel_p = f"{cls_name}/{p.name}"

                all_rows.append({
                    'rel_path': rel_p,
                    'fname': p.name,
                    'image_path': str(p.resolve()),
                    'class_id': int(cls_id),
                    'is_train': is_train,
                })
        df = pd.DataFrame(all_rows)

    # Thêm các cột dummy BBox để tương thích 100% với Dataset class ở Cell 3 mà không gây lỗi
    df['image_id'] = np.arange(len(df))
    df['bbox_x1'] = 0
    df['bbox_y1'] = 0
    df['bbox_x2'] = 99999
    df['bbox_y2'] = 99999

    train_df = df[df['is_train'] == 1].copy().reset_index(drop=True)
    train_df['split'] = 'train'
    train_df['row_id'] = np.arange(len(train_df))

    test_df = df[df['is_train'] == 0].copy().reset_index(drop=True)
    test_df['split'] = 'test'
    test_df['row_id'] = np.arange(len(test_df))

    return train_df, test_df


def split_train_val_by_class(df, val_fraction=0.10, seed=42):
    if not 0 < val_fraction < 1:
        raise ValueError('val_fraction phải nằm trong khoảng (0, 1)')
    rng = np.random.default_rng(seed)
    val_indices = []
    for _, group in df.groupby('class_id', sort=True):
        group_indices = group.index.to_numpy()
        if len(group_indices) <= 1:
            continue
        n_val = max(1, int(round(len(group_indices) * val_fraction)))
        n_val = min(n_val, len(group_indices) - 1)
        val_indices.extend(rng.choice(group_indices, size=n_val, replace=False).tolist())
    val_indices = np.array(sorted(val_indices), dtype=int)
    train_split = df.drop(index=val_indices).reset_index(drop=True)
    val_split = df.loc[val_indices].reset_index(drop=True)
    return train_split, val_split


# --- Tải dữ liệu SUN397 ---
class_names = load_class_names(DATASET_ROOT)
full_train_df, test_df = load_sun397_annotations(DATASET_ROOT, IMAGE_ROOT, class_names)
train_df, val_df = split_train_val_by_class(full_train_df, VAL_FRACTION, SEED)

if TRAIN_ON_FULL_TRAIN:
    _, monitor_val_df = split_train_val_by_class(
        full_train_df,
        FULL_TRAIN_MONITOR_FRACTION,
        SEED,
    )
    train_df = full_train_df.reset_index(drop=True)
    val_df = monitor_val_df.reset_index(drop=True)

candidate_rng = np.random.default_rng(SEED)
if CANDIDATE_POOL_FRACTION >= 1.0:
    candidate_pool_positions = np.arange(len(train_df), dtype=int)
else:
    candidate_pool_size = max(1, int(round(len(train_df) * CANDIDATE_POOL_FRACTION)))
    candidate_pool_positions = np.sort(
        candidate_rng.choice(len(train_df), size=candidate_pool_size, replace=False)
    )

candidate_df = train_df.iloc[candidate_pool_positions].reset_index(drop=True)
candidate_df.to_csv(OUT_PUT_DIR / 'candidate_pool.csv', index=False)
val_df.to_csv(OUT_PUT_DIR / 'validation_split.csv', index=False)

print('DATASET_ROOT:', DATASET_ROOT.resolve())
print('IMAGE_ROOT:', IMAGE_ROOT.resolve())
print('OUT_PUT_DIR:', OUT_PUT_DIR.resolve())
print(f'Standard train images: {len(full_train_df):,}')
print(f'Actual train images: {len(train_df):,}')
print(f'Validation images: {len(val_df):,}')
print(f'Train on full train: {TRAIN_ON_FULL_TRAIN}')
print(f'Standard test images: {len(test_df):,}')
print(f'Candidate pool: {len(candidate_df):,}/{len(train_df):,} actual-train images ({CANDIDATE_POOL_FRACTION:.0%})')
print(f'TACS selected contexts per query: {CONTEXT_TOP_K}')
print(f'TACS uses full candidate pool during training: {TACS_USE_FULL_CANDIDATE_POOL}')
print(f'TACS Gumbel tau: {TACS_GUMBEL_TAU}, policy weight: {TACS_POLICY_WEIGHT}')
print(f'BBox crop enabled: {USE_BBOX_CROP}')
print(f'Train label range: {int(train_df.class_id.min())}..{int(train_df.class_id.max())}')
print(f'Test label range: {int(test_df.class_id.min())}..{int(test_df.class_id.max())}')
display(train_df.head())
display(val_df.head())
display(candidate_df.head())

DEVICE: cuda
Sử dụng official partition: Training_01.txt và Testing_01.txt
DATASET_ROOT: /kaggle/input/datasets/hiuphmnhtrung/sun397/SUN397/SUN397
IMAGE_ROOT: /kaggle/input/datasets/hiuphmnhtrung/sun397/SUN397/SUN397
OUT_PUT_DIR: /kaggle/working/outputs/tacs_adaptive_diverse_sun_true_val_split
Standard train images: 19,850
Actual train images: 17,865
Validation images: 1,985
Train on full train: False
Standard test images: 19,850
Candidate pool: 3,573/17,865 actual-train images (20%)
TACS selected contexts per query: 3
TACS uses full candidate pool during training: True
TACS Gumbel tau: 0.1, policy weight: 1.0
BBox crop enabled: False
Train label range: 0..396
Test label range: 0..396


,rel_path,fname,image_path,class_id,is_train,image_id,bbox_x1,bbox_y1,bbox_x2,bbox_y2,split,row_id
0,a/abbey/sun_aqswjsnjlrfzzhiz.jpg,sun_aqswjsnjlrfzzhiz.jpg,/kaggle/input/datasets/hiuphmnhtrung/sun397/SU...,0,1,0,0,0,99999,99999,train,0
1,a/abbey/sun_aaoktempcmudsvna.jpg,sun_aaoktempcmudsvna.jpg,/kaggle/input/datasets/hiuphmnhtrung/sun397/SU...,0,1,1,0,0,99999,99999,train,1
2,a/abbey/sun_arxtvfkkycnqhfkj.jpg,sun_arxtvfkkycnqhfkj.jpg,/kaggle/input/datasets/hiuphmnhtrung/sun397/SU...,0,1,2,0,0,99999,99999,train,2
3,a/abbey/sun_axcfsjajikefppey.jpg,sun_axcfsjajikefppey.jpg,/kaggle/input/datasets/hiuphmnhtrung/sun397/SU...,0,1,3,0,0,99999,99999,train,3
4,a/abbey/sun_aizsoszlxxvbjhqd.jpg,sun_aizsoszlxxvbjhqd.jpg,/kaggle/input/datasets/hiuphmnhtrung/sun397/SU...,0,1,5,0,0,99999,99999,train,5


,rel_path,fname,image_path,class_id,is_train,image_id,bbox_x1,bbox_y1,bbox_x2,bbox_y2,split,row_id
0,a/abbey/sun_arsnqaftitwcpsog.jpg,sun_arsnqaftitwcpsog.jpg,/kaggle/input/datasets/hiuphmnhtrung/sun397/SU...,0,1,4,0,0,99999,99999,train,4
1,a/abbey/sun_axcwnadpsaegsupw.jpg,sun_axcwnadpsaegsupw.jpg,/kaggle/input/datasets/hiuphmnhtrung/sun397/SU...,0,1,21,0,0,99999,99999,train,21
2,a/abbey/sun_adbutqsfuzkgmrwy.jpg,sun_adbutqsfuzkgmrwy.jpg,/kaggle/input/datasets/hiuphmnhtrung/sun397/SU...,0,1,31,0,0,99999,99999,train,31
3,a/abbey/sun_akpgqjmkigtgyyhf.jpg,sun_akpgqjmkigtgyyhf.jpg,/kaggle/input/datasets/hiuphmnhtrung/sun397/SU...,0,1,36,0,0,99999,99999,train,36
4,a/abbey/sun_asgeghboyugsatii.jpg,sun_asgeghboyugsatii.jpg,/kaggle/input/datasets/hiuphmnhtrung/sun397/SU...,0,1,49,0,0,99999,99999,train,49


,rel_path,fname,image_path,class_id,is_train,image_id,bbox_x1,bbox_y1,bbox_x2,bbox_y2,split,row_id
0,a/abbey/sun_aakbdcgfpksytcwj.jpg,sun_aakbdcgfpksytcwj.jpg,/kaggle/input/datasets/hiuphmnhtrung/sun397/SU...,0,1,9,0,0,99999,99999,train,9
1,a/abbey/sun_afuhbsqqwerjshqj.jpg,sun_afuhbsqqwerjshqj.jpg,/kaggle/input/datasets/hiuphmnhtrung/sun397/SU...,0,1,17,0,0,99999,99999,train,17
2,a/abbey/sun_alqgvwyohunprcsh.jpg,sun_alqgvwyohunprcsh.jpg,/kaggle/input/datasets/hiuphmnhtrung/sun397/SU...,0,1,20,0,0,99999,99999,train,20
3,a/abbey/sun_argijfrtajgytjrs.jpg,sun_argijfrtajgytjrs.jpg,/kaggle/input/datasets/hiuphmnhtrung/sun397/SU...,0,1,22,0,0,99999,99999,train,22
4,a/abbey/sun_atbseznnhjnnmefq.jpg,sun_atbseznnhjnnmefq.jpg,/kaggle/input/datasets/hiuphmnhtrung/sun397/SU...,0,1,27,0,0,99999,99999,train,27


## 3. Dataset và DataLoader

Với full candidate pool, train dataset chỉ cần trả về query image và label. Candidate images được cache riêng trong candidate image bank; selector sẽ tính score với toàn bộ candidate pool trong train loop.


In [6]:
class SimpleImageDataset(Dataset):
    def __init__(self, annotations, transform, crop_bbox=False):
        self.annotations = annotations.reset_index(drop=True).copy()
        self.transform = transform
        self.crop_bbox = crop_bbox

    def __len__(self):
        return len(self.annotations)

    def _open_image(self, row):
        image = Image.open(row['image_path']).convert('RGB')
        if self.crop_bbox:
            left = max(0, int(row['bbox_x1']) - 1)
            top = max(0, int(row['bbox_y1']) - 1)
            right = min(image.width, max(left + 1, int(row['bbox_x2'])))
            bottom = min(image.height, max(top + 1, int(row['bbox_y2'])))
            image = image.crop((left, top, right, bottom))
        return image

    def __getitem__(self, index):
        row = self.annotations.iloc[index]
        image = self.transform(self._open_image(row))
        return image, int(index), row['fname'], int(row['class_id'])


class StanfordCarsSimilarityContextDataset(Dataset):
    def __init__(self, annotations, candidate_annotations, context_positions, context_scores, query_transform, context_transform, crop_bbox=False):
        self.annotations = annotations.reset_index(drop=True).copy()
        self.candidate_annotations = candidate_annotations.reset_index(drop=True).copy()
        self.context_positions = np.asarray(context_positions, dtype=np.int64)
        self.context_scores = np.asarray(context_scores, dtype=np.float32)
        if self.context_positions.ndim == 1:
            self.context_positions = self.context_positions[:, None]
        if self.context_scores.ndim == 1:
            self.context_scores = self.context_scores[:, None]
        self.query_transform = query_transform
        self.context_transform = context_transform
        self.crop_bbox = crop_bbox
        if len(self.context_positions) != len(self.annotations):
            raise ValueError('context_positions phai co cung do dai voi annotations')
        if len(self.context_scores) != len(self.annotations):
            raise ValueError('context_scores phai co cung do dai voi annotations')

    def __len__(self):
        return len(self.annotations)

    def set_epoch(self, epoch):
        pass

    def _open_image(self, row):
        image = Image.open(row['image_path']).convert('RGB')
        if self.crop_bbox:
            left = max(0, int(row['bbox_x1']) - 1)
            top = max(0, int(row['bbox_y1']) - 1)
            right = min(image.width, max(left + 1, int(row['bbox_x2'])))
            bottom = min(image.height, max(top + 1, int(row['bbox_y2'])))
            image = image.crop((left, top, right, bottom))
        return image

    def __getitem__(self, index):
        query_row = self.annotations.iloc[index]
        context_positions = self.context_positions[index]
        context_scores = self.context_scores[index]
        context_rows = [self.candidate_annotations.iloc[int(pos)] for pos in context_positions]
        query_image = self.query_transform(self._open_image(query_row))
        context_images = torch.stack([self.context_transform(self._open_image(context_row)) for context_row in context_rows], dim=0)
        context_labels = torch.tensor([int(context_row['class_id']) for context_row in context_rows], dtype=torch.long)
        return {
            'query_image': query_image,
            'context_images': context_images,
            'label': torch.tensor(int(query_row['class_id']), dtype=torch.long),
            'fname': query_row['fname'],
            'context_fnames': '|'.join(str(context_row['fname']) for context_row in context_rows),
            "context_labels": torch.tensor(context_labels, dtype=torch.long),
            'context_scores': torch.as_tensor(context_scores, dtype=torch.float32),
        }




class StanfordCarsTACSQueryDataset(Dataset):
    def __init__(self, annotations, transform, crop_bbox=False):
        self.annotations = annotations.reset_index(drop=True).copy()
        self.transform = transform
        self.crop_bbox = crop_bbox

    def __len__(self):
        return len(self.annotations)

    def _open_image(self, row):
        image = Image.open(row['image_path']).convert('RGB')
        if self.crop_bbox:
            left = max(0, int(row['bbox_x1']) - 1)
            top = max(0, int(row['bbox_y1']) - 1)
            right = min(image.width, max(left + 1, int(row['bbox_x2'])))
            bottom = min(image.height, max(top + 1, int(row['bbox_y2'])))
            image = image.crop((left, top, right, bottom))
        return image

    def __getitem__(self, index):
        row = self.annotations.iloc[index]
        return {
            'query_image': self.transform(self._open_image(row)),
            'label': torch.tensor(int(row['class_id']), dtype=torch.long),
            'fname': row['fname'],
            'row_id': torch.tensor(int(row['row_id']), dtype=torch.long),
        }


## 4. Adaptive diversity-aware selector

MMR chon tuan tu tu toan candidate pool bang utility da chuan hoa tru diversity penalty (do tuong dong voi context da chon). Attention pooling nhan mask de bo qua context bi loai bo.


In [7]:
from pathlib import Path
import gc
import torch


def resolve_dinov3_weights():
    path = Path(DINOV3_WEIGHTS)
    if not path.is_file():
        raise FileNotFoundError(f'Không tìm thấy file weight: {path}')
    return str(path)


def load_dinov3_model(repo_dir, model_name, weights_path):
    repo_dir = Path(repo_dir)
    if not repo_dir.is_dir():
        raise FileNotFoundError(f'Không tìm thấy thư mục mã nguồn: {repo_dir}')
    weights_path = Path(weights_path)
    if not weights_path.is_file():
        raise FileNotFoundError(f'Không tìm thấy file weight: {weights_path}')
    return torch.hub.load(str(repo_dir), model_name, source='local', weights=str(weights_path))


def load_dinov3_backbone():
    return load_dinov3_model(DINOV3_REPO_DIR, DINOV3_MODEL_NAME, resolve_dinov3_weights())


def ensure_dinov3_assets():
    resolve_dinov3_weights()
    repo_dir = Path(DINOV3_REPO_DIR)
    if not repo_dir.is_dir():
        raise FileNotFoundError(f'Không tìm thấy thư mục mã nguồn: {repo_dir}')
    print('DINOv3 assets đã sẵn sàng từ Kaggle Input.')


def extract_cls_features(backbone, images):
    features = backbone(images)
    if isinstance(features, dict):
        if 'x_norm_clstoken' in features:
            return features['x_norm_clstoken']
        if 'x_norm_cls_token' in features:
            return features['x_norm_cls_token']
        if 'x_norm_patchtokens' in features:
            return features['x_norm_patchtokens'].mean(dim=1)
        raise KeyError(f'Khong tim thay CLS token trong output keys: {list(features)}')
    return features[:, 0] if torch.is_tensor(features) and features.ndim == 3 else features


def extract_frozen_dino_embeddings(annotations, backbone, desc):
    dataset = SimpleImageDataset(annotations, eval_transform, crop_bbox=USE_BBOX_CROP)
    loader = DataLoader(dataset, batch_size=RETRIEVAL_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True,persistent_workers=True if NUM_WORKERS > 0 else False)
    embeddings = []
    backbone.eval()
    for parameter in backbone.parameters():
        parameter.requires_grad_(False)
    with torch.inference_mode():
        for images, *_ in tqdm(loader, desc=desc, leave=False):
            images = images.to(DEVICE, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=DEVICE.type == 'cuda'):
                features = extract_cls_features(backbone, images)
            features = F.normalize(features.float(), dim=1)
            embeddings.append(features.cpu())
    return torch.cat(embeddings, dim=0)




class DinoV3FrozenSimilarityContextClassifier(nn.Module):
    def __init__(self, repo_dir, model_name, weights_path, num_classes, feature_dim=384, allow_feature_fallback=False):
        super().__init__()
        self.backbone = load_dinov3_model(repo_dir, model_name, weights_path)
        self.feature_dim = feature_dim
        self.allow_feature_fallback = allow_feature_fallback
        self.active_fusion_mode = 'topk_token_attention'
        self.context_proj = nn.Linear(feature_dim, feature_dim)
        self.query_proj = nn.Linear(feature_dim, feature_dim)
        self.context_attention = nn.Linear(feature_dim, 1)
        self.fusion_norm = nn.LayerNorm(feature_dim)
        self.token_head = nn.Sequential(nn.Dropout(p=TOKEN_HEAD_DROPOUT), nn.Linear(feature_dim, num_classes))
        self.feature_fusion = nn.Sequential(nn.Linear(feature_dim * 2, feature_dim), nn.GELU(), nn.Dropout(p=TOKEN_HEAD_DROPOUT), nn.Linear(feature_dim, num_classes))
        for param in self.backbone.parameters():
            param.requires_grad_(False)

    def _prefix_token_count(self):
        if hasattr(self.backbone, 'num_prefix_tokens'):
            return int(self.backbone.num_prefix_tokens)
        storage_tokens = getattr(self.backbone, 'n_storage_tokens', None)
        if storage_tokens is None:
            storage_tokens = getattr(self.backbone, 'num_register_tokens', 0)
        return 1 + int(storage_tokens or 0)

    def _patch_grid(self, images):
        patch_size = getattr(self.backbone, 'patch_size', 16)
        if isinstance(patch_size, (tuple, list)):
            patch_h, patch_w = int(patch_size[0]), int(patch_size[1])
        else:
            patch_h = patch_w = int(patch_size)
        return images.shape[-2] // patch_h, images.shape[-1] // patch_w

    def _unwrap_tokens(self, prepared):
        if torch.is_tensor(prepared):
            return prepared
        if isinstance(prepared, (tuple, list)):
            for item in prepared:
                if torch.is_tensor(item):
                    return item
        if isinstance(prepared, dict):
            for key in ('x', 'tokens', 'last_hidden_state'):
                if key in prepared and torch.is_tensor(prepared[key]):
                    return prepared[key]
            for item in prepared.values():
                if torch.is_tensor(item):
                    return item
        raise TypeError(f'Cannot unwrap DINO token output of type {type(prepared)}')

    def _prepare_tokens_with_rope(self, images):
        try:
            prepared = self.backbone.prepare_tokens_with_masks(images, None)
        except TypeError:
            prepared = self.backbone.prepare_tokens_with_masks(images)
        tokens = self._unwrap_tokens(prepared)
        rope = None
        if hasattr(self.backbone, 'rope_embed'):
            grid_h, grid_w = self._patch_grid(images)
            try:
                rope = self.backbone.rope_embed(H=grid_h, W=grid_w)
            except TypeError:
                rope = self.backbone.rope_embed(grid_h, grid_w)
        return tokens, rope

    def _repeat_rope_for_query_context(self, rope):
        if rope is None or not torch.is_tensor(rope):
            return rope
        try:
            return rope.repeat_interleave(2, dim=-2)
        except RuntimeError:
            return rope

    def _run_blocks(self, tokens, rope):
        for block in self.backbone.blocks:
            if rope is None:
                tokens = block(tokens)
                continue
            try:
                tokens = block(tokens, rope=rope)
            except TypeError:
                try:
                    tokens = block(tokens, rope)
                except TypeError:
                    tokens = block(tokens)
        if hasattr(self.backbone, 'norm'):
            tokens = self.backbone.norm(tokens)
        return tokens

    def _forward_token_concat_features(self, query_images, context_images):
        query_tokens, query_rope = self._prepare_tokens_with_rope(query_images)
        context_tokens, _ = self._prepare_tokens_with_rope(context_images)
        prefix_tokens = self._prefix_token_count()
        pair_tokens = torch.cat([query_tokens[:, :prefix_tokens], query_tokens[:, prefix_tokens:], context_tokens[:, prefix_tokens:]], dim=1)
        pair_rope = self._repeat_rope_for_query_context(query_rope)
        pair_tokens = self._run_blocks(pair_tokens, pair_rope)
        return pair_tokens[:, 0]

    def _pool_backbone_features(self, images):
        features = self.backbone.forward_features(images)
        if isinstance(features, dict):
            if 'x_norm_clstoken' in features:
                return features['x_norm_clstoken']
            if 'x_norm_patchtokens' in features:
                return features['x_norm_patchtokens'].mean(dim=1)
            for item in features.values():
                if torch.is_tensor(item):
                    return item[:, 0] if item.ndim == 3 else item
        if torch.is_tensor(features):
            return features[:, 0] if features.ndim == 3 else features
        raise TypeError(f'Cannot pool DINO features of type {type(features)}')

    def _forward_feature_fusion(self, query_images, context_images):
        if context_images.ndim == 5:
            context_images = context_images[:, 0]
        query_features = self._pool_backbone_features(query_images)
        context_features = self._pool_backbone_features(context_images)
        self.active_fusion_mode = 'feature_fusion'
        return self.feature_fusion(torch.cat([query_features, context_features], dim=1))

    def _forward_topk_token_attention(self, query_images, context_images, context_scores=None, context_mask=None):
        if context_images.ndim == 4:
            context_images = context_images.unsqueeze(1)
        batch_size, num_contexts, channels, height, width = context_images.shape
        query_repeated = query_images.unsqueeze(1).expand(-1, num_contexts, -1, -1, -1).reshape(batch_size * num_contexts, channels, height, width)
        context_flat = context_images.reshape(batch_size * num_contexts, channels, height, width)
        pair_features = self._forward_token_concat_features(query_repeated, context_flat).reshape(batch_size, num_contexts, self.feature_dim)
        if context_mask is None:
            context_mask = torch.ones(batch_size, num_contexts, device=pair_features.device, dtype=torch.bool)
        else:
            context_mask = context_mask.to(device=pair_features.device, dtype=torch.bool)
            if context_mask.shape != (batch_size, num_contexts) or not context_mask.any(dim=1).all():
                raise ValueError('Each query must have at least one active context.')
        mask_values = context_mask.unsqueeze(-1).to(pair_features.dtype)
        query_summary = (pair_features * mask_values).sum(dim=1) / mask_values.sum(dim=1).clamp_min(1.0)
        attention_hidden = torch.tanh(self.context_proj(pair_features) + self.query_proj(query_summary).unsqueeze(1))
        attention_logits = self.context_attention(attention_hidden).squeeze(-1)
        if context_scores is not None:
            if context_scores.ndim == 1:
                context_scores = context_scores.unsqueeze(1)
            attention_logits = attention_logits + RETRIEVAL_SCORE_WEIGHT * context_scores.to(attention_logits.device, dtype=attention_logits.dtype)
        attention_logits = attention_logits.masked_fill(~context_mask, torch.finfo(attention_logits.dtype).min)
        attention_weights = torch.softmax(attention_logits, dim=1)
        fused_features = torch.sum(pair_features * attention_weights.unsqueeze(-1), dim=1)
        fused_features = self.fusion_norm(fused_features)
        self.active_fusion_mode = 'topk_token_attention'
        return fused_features, attention_weights

    def forward(self, query_images, context_images, context_scores=None, return_features=False, context_mask=None):
        try:
            fused_features, attention_weights = self._forward_topk_token_attention(
                query_images, context_images, context_scores=context_scores, context_mask=context_mask
            )
            logits = self.token_head(fused_features)
            if return_features:
                return logits, F.normalize(fused_features.float(), dim=1), attention_weights
            return logits
        except Exception as exc:
            if self.allow_feature_fallback:
                print(f'Top-k token attention failed, fallback to feature fusion: {type(exc).__name__}: {exc}')
                logits = self._forward_feature_fusion(query_images, context_images)
                if return_features:
                    return logits, F.normalize(logits.float(), dim=1), None
                return logits
            raise RuntimeError('Top-k token attention failed. Keep allow_feature_fallback=False for debugging.') from exc

class DinoV3GatedTopKContextClassifier(DinoV3FrozenSimilarityContextClassifier):
    def __init__(self, repo_dir, model_name, weights_path, num_classes, feature_dim=384, allow_feature_fallback=False):
        super().__init__(
            repo_dir=repo_dir,
            model_name=model_name,
            weights_path=weights_path,
            num_classes=num_classes,
            feature_dim=feature_dim,
            allow_feature_fallback=allow_feature_fallback,
        )
        self.query_norm = nn.LayerNorm(feature_dim)
        self.context_norm = nn.LayerNorm(feature_dim)
        self.final_norm = nn.LayerNorm(feature_dim)

        self.query_head = nn.Sequential(
            nn.Dropout(p=TOKEN_HEAD_DROPOUT),
            nn.Linear(feature_dim, num_classes),
        )

        self.fusion_gate = nn.Sequential(
            nn.Linear(feature_dim * 4, feature_dim),
            nn.GELU(),
            nn.Dropout(p=QUERY_CONTEXT_GATE_DROPOUT),
            nn.Linear(feature_dim, 1),
        )
        nn.init.constant_(self.fusion_gate[-1].bias, QUERY_CONTEXT_GATE_BIAS)
        self.active_fusion_mode = "query_plus_gated_topk_context"

    def forward(self, query_images, context_images, context_scores=None, return_features=False, context_mask=None):
        try:
            context_features, attention_weights = self._forward_topk_token_attention(
                query_images,
                context_images,
                context_scores=context_scores,
                context_mask=context_mask,
            )
            query_features = self._pool_backbone_features(query_images)

            query_features = self.query_norm(query_features)
            context_features = self.context_norm(context_features)

            gate_input = torch.cat(
                [
                    query_features,
                    context_features,
                    query_features * context_features,
                    torch.abs(query_features - context_features),
                ],
                dim=1,
            )
            gate = torch.sigmoid(self.fusion_gate(gate_input))

            query_logits = self.query_head(query_features)
            context_logits = self.token_head(context_features)
            logits = query_logits + gate * context_logits

            fused_features = self.final_norm(query_features + gate * context_features)
            self.active_fusion_mode = "query_plus_gated_topk_context"

            if return_features:
                return logits, F.normalize(fused_features.float(), dim=1), attention_weights
            return logits

        except Exception as exc:
            if self.allow_feature_fallback:
                print(f"Gated fusion failed, fallback to parent model: {type(exc).__name__}: {exc}")
                return super().forward(query_images, context_images, context_scores=context_scores, return_features=return_features)
            raise




class TACSSelector(nn.Module):
    def __init__(self, temperature=1.0, normalize=True):
        super().__init__()
        self.temperature = float(temperature)
        self.normalize = bool(normalize)
        self.logit_scale = nn.Parameter(torch.tensor(1.0))

    def score(self, query_features, candidate_features):
        query = query_features.float()
        candidates = candidate_features.float()
        if self.normalize:
            query = F.normalize(query, dim=-1)
            candidates = F.normalize(candidates, dim=-1)
        scores = query @ candidates.T
        return scores * self.logit_scale.clamp(0.01, 100.0) / max(self.temperature, 1e-6)


class DinoV3TACSClassifier(DinoV3GatedTopKContextClassifier):
    def __init__(self, repo_dir, model_name, weights_path, num_classes, feature_dim=384, allow_feature_fallback=False):
        super().__init__(
            repo_dir=repo_dir,
            model_name=model_name,
            weights_path=weights_path,
            num_classes=num_classes,
            feature_dim=feature_dim,
            allow_feature_fallback=allow_feature_fallback,
        )
        self.selector = TACSSelector(
            temperature=TACS_SELECTOR_TEMPERATURE,
            normalize=True,
        )
        self.active_fusion_mode = 'tacs_full_pool_shared_backbone_selector'

    def selector_encode_images(self, images):
        return self._pool_backbone_features(images)

    def score_full_candidate_pool(self, query_images, candidate_feature_bank):
        query_features = self.selector_encode_images(query_images)
        selector_scores = self.selector.score(query_features, candidate_feature_bank.to(query_features.device))
        return selector_scores, query_features

    def select_full_pool(self, query_images, candidate_feature_bank, use_gumbel=True):
        selector_scores, query_features = self.score_full_candidate_pool(query_images, candidate_feature_bank)
        selection = 'gumbel' if self.training and use_gumbel else 'greedy'
        selected_indices, selection_weights, selected_mmr_scores, _ = select_diverse_indices(
            selector_scores, candidate_feature_bank, CONTEXT_TOP_K, selection=selection
        )
        return {
            'selector_scores': selector_scores,
            'selection_weights': selection_weights,
            'selected_indices': selected_indices,
            'selected_mmr_scores': selected_mmr_scores,
            'query_features': query_features,
            'selector_entropy': torch.distributions.Categorical(logits=selector_scores.float()).entropy().mean(),
        }

    def forward_selected_context(self, query_images, selected_context_images, return_features=False, context_mask=None):
        if selected_context_images.ndim == 4:
            selected_context_images = selected_context_images.unsqueeze(1)
        if selected_context_images.ndim != 5:
            raise ValueError('Contexts must have shape [B, K, C, H, W] or [B, C, H, W].')
        return super().forward(query_images, selected_context_images, context_scores=None,
                               return_features=return_features, context_mask=context_mask)

    def forward_query_only(self, query_images):
        query_features = self._pool_backbone_features(query_images)
        if hasattr(self, 'query_norm') and hasattr(self, 'query_head'):
            query_features = self.query_norm(query_features)
            return self.query_head(query_features)
        return self.token_head(query_features)


def build_candidate_image_bank(annotations, transform, device, dtype):
    dataset = SimpleImageDataset(annotations, transform, crop_bbox=USE_BBOX_CROP)
    loader = DataLoader(dataset, batch_size=RETRIEVAL_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    images_bank = []
    labels = []
    fnames = []
    for images, labels_batch, fnames_batch, *_ in tqdm(loader, desc='build candidate image bank', leave=False):
        images_bank.append(images.to(device=device, dtype=dtype, non_blocking=True).contiguous())
        labels.extend([int(x) for x in labels_batch.tolist()])
        fnames.extend(fnames_batch)
    return torch.cat(images_bank, dim=0), torch.tensor(labels, dtype=torch.long, device=device), fnames


@torch.no_grad()
def refresh_candidate_feature_bank(network, candidate_image_bank, batch_size=CANDIDATE_FEATURE_BANK_BATCH_SIZE):
    was_training = network.training
    network.eval()
    features = []
    with torch.inference_mode():
        for start in tqdm(range(0, len(candidate_image_bank), batch_size), desc='refresh full candidate feature bank', leave=False):
            end = min(start + batch_size, len(candidate_image_bank))
            images = candidate_image_bank[start:end].to(DEVICE, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=DEVICE.type == 'cuda'):
                feats = network.selector_encode_images(images)
            features.append(feats.float().detach().cpu())
    if was_training:
        network.train()
    return torch.cat(features, dim=0)


ensure_dinov3_assets()

train_dataset = StanfordCarsTACSQueryDataset(train_df, train_transform, crop_bbox=USE_BBOX_CROP)
val_dataset = StanfordCarsTACSQueryDataset(val_df, eval_transform, crop_bbox=USE_BBOX_CROP)
test_dataset = StanfordCarsTACSQueryDataset(test_df, eval_transform, crop_bbox=USE_BBOX_CROP)

loader_kwargs = {'num_workers': NUM_WORKERS, 'pin_memory': torch.cuda.is_available()}
if NUM_WORKERS > 0:
    loader_kwargs['persistent_workers'] = True
    loader_kwargs['prefetch_factor'] = 2

train_generator = torch.Generator()
train_generator.manual_seed(SEED)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, generator=train_generator, **loader_kwargs)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)

model = DinoV3TACSClassifier(
    DINOV3_REPO_DIR,
    DINOV3_MODEL_NAME,
    resolve_dinov3_weights(),
    NUM_CLASSES,
    FEATURE_DIM,
    allow_feature_fallback=False,
).to(DEVICE)

candidate_image_bank, candidate_labels_tensor, candidate_fnames = build_candidate_image_bank(
    candidate_df,
    eval_transform,
    device=torch.device(CANDIDATE_IMAGE_BANK_DEVICE),
    dtype=CANDIDATE_IMAGE_BANK_DTYPE,
)

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
backbone_params = [p for name, p in model.named_parameters() if name.startswith('backbone.')]
selector_params = [p for name, p in model.named_parameters() if name.startswith('selector.')]
head_params = [p for name, p in model.named_parameters() if not name.startswith('backbone.') and not name.startswith('selector.')]
optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': BACKBONE_LR, 'weight_decay': BACKBONE_WEIGHT_DECAY, 'name': 'backbone'},
    {'params': selector_params, 'lr': TACS_SELECTOR_LR, 'weight_decay': TACS_SELECTOR_WEIGHT_DECAY, 'name': 'selector'},
    {'params': head_params, 'lr': HEAD_LR, 'weight_decay': HEAD_WEIGHT_DECAY, 'name': 'head'},
])

warmup_epochs = max(1, min(WARMUP_EPOCHS, EPOCHS))
main_epochs = max(1, EPOCHS - warmup_epochs)
warmup_scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.2, end_factor=1.0, total_iters=warmup_epochs)
cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=main_epochs, eta_min=1e-7)
scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warmup_epochs])
scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == 'cuda')

checkpoint_path = OUT_PUT_DIR / 'final_tacs_full_pool_dinov3_vits16_sun397.pt'
last_checkpoint_path = OUT_PUT_DIR / 'checkpoint_last.pt'
best_checkpoint_path = OUT_PUT_DIR / 'best_val_checkpoint.pt'

torch.save({
    'candidate_pool_positions': torch.as_tensor(candidate_pool_positions, dtype=torch.long),
    'seed': SEED,
    'candidate_pool_fraction': CANDIDATE_POOL_FRACTION,
    'use_bbox_crop': USE_BBOX_CROP,
    'selector_mode': 'full_candidate_pool_shared_backbone',
    'candidate_image_bank_device': CANDIDATE_IMAGE_BANK_DEVICE,
    'candidate_image_bank_dtype': str(CANDIDATE_IMAGE_BANK_DTYPE),
}, OUT_PUT_DIR / 'tacs_selector_cache.pt')

sample = train_dataset[0]
print('Saved TACS selector cache:', (OUT_PUT_DIR / 'tacs_selector_cache.pt').resolve())
print('Candidate image bank:', candidate_image_bank.shape, candidate_image_bank.device, candidate_image_bank.dtype)
print('Sample query tensor:', sample['query_image'].shape)
print('Candidate pool size used per query:', len(candidate_df))
print('Initial fusion mode:', model.active_fusion_mode)
print(model)


DINOv3 assets đã sẵn sàng từ Kaggle Input.
Downloading: "file:///kaggle/input/datasets/sinhnguync/dinov3/dinov3_vits16_pretrain_lvd1689m-08c60483.pth" to /root/.cache/torch/hub/checkpoints/dinov3_vits16_pretrain_lvd1689m-08c60483.pth


100%|██████████| 82.5M/82.5M [00:01<00:00, 63.0MB/s]


build candidate image bank:   0%|          | 0/7 [00:00<?, ?it/s]

Saved TACS selector cache: /kaggle/working/outputs/tacs_adaptive_diverse_sun_true_val_split/tacs_selector_cache.pt
Candidate image bank: torch.Size([3573, 3, 384, 384]) cuda:0 torch.float16
Sample query tensor: torch.Size([3, 384, 384])
Candidate pool size used per query: 3573
Initial fusion mode: tacs_full_pool_shared_backbone_selector
DinoV3TACSClassifier(
  (backbone): DinoVisionTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
      (norm): Identity()
    )
    (rope_embed): RopePositionEmbedding()
    (blocks): ModuleList(
      (0-11): 12 x SelfAttentionBlock(
        (norm1): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
        (attn): SelfAttention(
          (qkv): LinearKMaskedBias(in_features=384, out_features=1152, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=384, out_features=384, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
    

/tmp/ipykernel_163/2411397457.py:453: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == 'cuda')


## 5. Train voi K thay doi

Moi batch train random K trong [1, 3]. Validation dung MMR tao toi da ba context; adaptive threshold chon so context. Threshold dung cho test duoc chon tren validation cua best checkpoint.


In [8]:
from sklearn.metrics import f1_score


def macro_f1_from_predictions(labels, predictions):
    return float(f1_score(labels, predictions, average='macro', zero_division=0))


def supervised_contrastive_loss(features, labels, temperature=0.10):
    features = F.normalize(features.float(), dim=1)
    labels = labels.view(-1, 1)
    mask = torch.eq(labels, labels.T).float().to(features.device)
    logits = torch.matmul(features, features.T) / temperature
    logits = logits - logits.max(dim=1, keepdim=True).values.detach()
    logits_mask = torch.ones_like(mask) - torch.eye(mask.size(0), device=features.device)
    positives_mask = mask * logits_mask
    positive_counts = positives_mask.sum(dim=1)
    valid = positive_counts > 0
    if not torch.any(valid):
        return features.sum() * 0.0
    exp_logits = torch.exp(logits) * logits_mask
    log_prob = logits - torch.log(exp_logits.sum(dim=1, keepdim=True).clamp_min(1e-12))
    mean_log_prob_pos = (positives_mask * log_prob).sum(dim=1) / positive_counts.clamp_min(1.0)
    return -mean_log_prob_pos[valid].mean()


def set_backbone_trainable(network, trainable):
    for name, param in network.named_parameters():
        if name.startswith('backbone.'):
            param.requires_grad_(trainable)
    if hasattr(network, 'backbone'):
        network.backbone.train(mode=trainable)


def current_lrs(opt):
    return {group.get('name', f'group_{index}'): group['lr'] for index, group in enumerate(opt.param_groups)}


def move_query_batch_to_device(batch):
    return {
        'query_images': batch['query_image'].to(DEVICE, non_blocking=True),
        'labels': batch['label'].to(DEVICE, non_blocking=True),
        'fname': batch['fname'],
    }


def add_context_label_prior(logits, context_labels, context_scores):
    if not USE_CONTEXT_LABEL_PRIOR:
        return logits
    if context_labels.ndim == 1:
        context_labels = context_labels.unsqueeze(1)
    if context_scores.ndim == 1:
        context_scores = context_scores.unsqueeze(1)
    weights = torch.softmax(context_scores / CONTEXT_LABEL_PRIOR_TEMPERATURE, dim=1)
    prior_logits = torch.zeros_like(logits)
    prior_logits.scatter_add_(dim=1, index=context_labels.long(), src=weights.to(logits.dtype))
    return logits + CONTEXT_LABEL_PRIOR_WEIGHT * prior_logits


def gather_bank_images(indices):
    return candidate_image_bank[indices.detach().to(candidate_image_bank.device)].to(DEVICE, non_blocking=True).float()


def select_bank_images_from_weights(selection_weights):
    """Forward selects real images; straight-through weights pass gradients to selector."""
    bank = candidate_image_bank.to(DEVICE, non_blocking=True)
    bank_flat = bank.flatten(1)
    weights = selection_weights.to(device=bank_flat.device, dtype=bank_flat.dtype)
    selected_flat = torch.matmul(weights, bank_flat)  # [B, K, pixels]
    return selected_flat.reshape(weights.size(0), weights.size(1), *bank.shape[1:]).float()


def adaptive_context_mask(selected_mmr_scores, threshold):
    gaps = selected_mmr_scores[:, :1] - selected_mmr_scores
    keep = gaps <= float(threshold)
    keep[:, 0] = True
    for rank in range(1, keep.size(1)):
        keep[:, rank] &= keep[:, rank - 1]
    return keep


def select_diverse_indices(selector_scores, candidate_feature_bank, max_k, selection='greedy'):
    scores = selector_scores.float()
    relevance = (scores - scores.mean(1, keepdim=True)) / scores.std(
        1, keepdim=True, unbiased=False
    ).clamp_min(1e-6)
    features = F.normalize(candidate_feature_bank.to(scores.device).float(), dim=1)
    max_k = min(int(max_k), scores.size(1))
    chosen = torch.zeros_like(scores, dtype=torch.bool)
    redundancy = torch.zeros_like(scores)
    indices, weights, mmr_scores, log_probs = [], [], [], []
    for _ in range(max_k):
        mmr = (relevance - DIVERSITY_WEIGHT * redundancy).masked_fill(
            chosen, torch.finfo(relevance.dtype).min
        )
        if selection == 'gumbel':
            weight = F.gumbel_softmax(mmr, tau=TACS_GUMBEL_TAU, hard=True, dim=1)
            index = weight.argmax(1)
        elif selection == 'sample':
            dist = torch.distributions.Categorical(logits=mmr)
            index = dist.sample()
            log_probs.append(dist.log_prob(index))
            weight = F.one_hot(index, scores.size(1)).to(scores.dtype)
        else:
            index = mmr.argmax(1)
            weight = F.one_hot(index, scores.size(1)).to(scores.dtype)
        indices.append(index)
        weights.append(weight)
        mmr_scores.append(mmr.gather(1, index[:, None]).squeeze(1))
        chosen = chosen.scatter(1, index[:, None], True)
        similarity = features @ features[index].T
        redundancy = torch.maximum(redundancy, similarity.T.clamp_min(0.0))
    log_probs = torch.stack(log_probs, dim=1) if log_probs else None
    return torch.stack(indices, 1), torch.stack(weights, 1), torch.stack(mmr_scores, 1), log_probs


def sample_policy_topk(selector_scores, candidate_feature_bank, k):
    actions, _weights, _mmr, log_terms = select_diverse_indices(
        selector_scores,
        candidate_feature_bank,
        k,
        selection='sample',
    )
    return actions, log_terms


def gather_bank_labels(indices):
    return candidate_labels_tensor[indices.detach().to(candidate_labels_tensor.device)].to(DEVICE, non_blocking=True)


def train_one_epoch(network, loader, epoch, candidate_feature_bank, backbone_trainable=True):
    network.train()
    set_backbone_trainable(network, backbone_trainable)
    bank = candidate_feature_bank.to(DEVICE, non_blocking=True)
    running_loss = running_ce = running_supcon = running_policy = running_reward = running_entropy = 0.0
    running_same = running_correct = running_total = running_contexts = 0
    all_predictions, all_labels = [], []
    progress = tqdm(loader, desc=f'Epoch {epoch:03d} [adaptive diverse train]', leave=False)
    for batch in progress:
        moved = move_query_batch_to_device(batch)
        optimizer.zero_grad(set_to_none=True)
        active_k = random.randint(1, CONTEXT_TOP_K)
        mask = (torch.arange(CONTEXT_TOP_K, device=DEVICE)[None, :] < active_k)
        mask = mask.expand(moved['labels'].size(0), -1)
        with torch.cuda.amp.autocast(enabled=DEVICE.type == 'cuda'):
            details = network.select_full_pool(moved['query_images'], bank, use_gumbel=True)
            images = select_bank_images_from_weights(details['selection_weights'])
            logits, features, _ = network.forward_selected_context(
                moved['query_images'], images, context_mask=mask, return_features=True
            )
            ce = criterion(logits, moved['labels'])
            supcon = supervised_contrastive_loss(features, moved['labels'], temperature=SUPCON_TEMPERATURE)
            actions, policy_terms = sample_policy_topk(details['selector_scores'].float(), bank, CONTEXT_TOP_K)
            log_prob = policy_terms[:, :active_k].sum(1)
            entropy = torch.distributions.Categorical(logits=details['selector_scores'].float()).entropy().mean()
            with torch.no_grad():
                query_logits = network.forward_query_only(moved['query_images'])
                policy_images = gather_bank_images(actions)
                policy_logits = network.forward_selected_context(
                    moved['query_images'], policy_images, context_mask=mask, return_features=False
                )
                qloss = F.cross_entropy(query_logits.float(), moved['labels'], reduction='none')
                closs = F.cross_entropy(policy_logits.float(), moved['labels'], reduction='none')
                rewards = (qloss - closs).detach()
                advantages = ((rewards - rewards.mean()) / rewards.std(unbiased=False).clamp_min(1e-6)).detach()
            policy_loss = -(log_prob * advantages).mean()
            loss = ce + SUPCON_WEIGHT * supcon + TACS_POLICY_WEIGHT * policy_loss - TACS_ENTROPY_WEIGHT * entropy
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(network.parameters(), GRAD_CLIP_NORM)
        scaler.step(optimizer); scaler.update()
        pred = logits.argmax(1)
        n = moved['labels'].size(0)
        labels = gather_bank_labels(details['selected_indices'])
        same = labels[:, :active_k].eq(moved['labels'][:, None])
        running_same += same.any(1).sum().item()
        running_contexts += active_k * n
        running_correct += pred.eq(moved['labels']).sum().item()
        running_total += n
        running_loss += loss.item() * n; running_ce += ce.item() * n
        running_supcon += supcon.item() * n; running_policy += policy_loss.item() * n
        running_reward += rewards.mean().item() * n; running_entropy += entropy.item() * n
        all_predictions.extend(pred.detach().cpu().tolist())
        all_labels.extend(moved['labels'].detach().cpu().tolist())
        progress.set_postfix(loss=running_loss/max(running_total,1), acc=running_correct/max(running_total,1), k=active_k)
    den=max(running_total,1)
    return {'loss':running_loss/den,'ce_loss':running_ce/den,'supcon_loss':running_supcon/den,
            'policy_loss':running_policy/den,'reward':running_reward/den,'selector_entropy':running_entropy/den,
            'accuracy':running_correct/den,'macro_f1':macro_f1_from_predictions(all_labels,all_predictions),
            'selected_same_class_rate':running_same/den,'mean_context_count':running_contexts/den}


@torch.no_grad()
def evaluate_one_epoch(network, loader, candidate_feature_bank, desc='val'):
    network.eval()
    bank = candidate_feature_bank.to(DEVICE, non_blocking=True)
    loss_sum = correct = total = same_sum = entropy_sum = context_sum = 0.0
    predictions, labels_all = [], []
    for batch in tqdm(loader, desc=desc, leave=False):
        moved = move_query_batch_to_device(batch)
        with torch.cuda.amp.autocast(enabled=DEVICE.type == 'cuda'):
            details = network.select_full_pool(moved['query_images'], bank, use_gumbel=False)
            mask = adaptive_context_mask(details['selected_mmr_scores'], ADAPTIVE_SCORE_GAP_THRESHOLD)
            images = gather_bank_images(details['selected_indices'])
            logits = network.forward_selected_context(
                moved['query_images'], images, context_mask=mask, return_features=False
            )
            loss = criterion(logits, moved['labels'])
        pred=logits.argmax(1); n=moved['labels'].size(0)
        ctx_labels=gather_bank_labels(details['selected_indices'])
        same_sum += (ctx_labels.eq(moved['labels'][:,None]) & mask).any(1).sum().item()
        context_sum += mask.sum().item()
        entropy_sum += details['selector_entropy'].item()*n
        loss_sum += loss.item()*n; correct += pred.eq(moved['labels']).sum().item(); total += n
        predictions.extend(pred.detach().cpu().tolist()); labels_all.extend(moved['labels'].detach().cpu().tolist())
    den=max(total,1)
    return {'loss':loss_sum/den,'accuracy':correct/den,
            'macro_f1':macro_f1_from_predictions(labels_all,predictions),
            'selected_same_class_rate':same_sum/den,'selector_entropy':entropy_sum/den,
            'mean_context_count':context_sum/den}


def build_checkpoint_payload(epoch, train_metrics, val_metrics, best_val_loss, best_epoch, history):
    return {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'class_names': class_names,
        'metrics': {
            'train_loss': train_metrics['loss'],
            'train_ce_loss': train_metrics.get('ce_loss'),
            'train_supcon_loss': train_metrics.get('supcon_loss'),
            'train_policy_loss': train_metrics.get('policy_loss'),
            'train_reward': train_metrics.get('reward'),
            'train_selector_entropy': train_metrics.get('selector_entropy'),
            'train_selected_same_class_any_context_rate': train_metrics.get('selected_same_class_rate'),
            'train_mean_context_count': train_metrics.get('mean_context_count'),
            'train_accuracy': train_metrics['accuracy'],
            'train_macro_f1': train_metrics['macro_f1'],
            'val_loss': val_metrics['loss'],
            'val_accuracy': val_metrics['accuracy'],
            'val_macro_f1': val_metrics['macro_f1'],
            'val_selected_same_class_any_context_rate': val_metrics.get('selected_same_class_rate'),
            'val_mean_context_count': val_metrics.get('mean_context_count'),
            'val_selector_entropy': val_metrics.get('selector_entropy'),
            'best_val_loss': best_val_loss,
            'best_epoch': best_epoch,
        },
        'history': history,
        'config': {
            'dataset_root': str(DATASET_ROOT),
            'method': 'TACS',
            'selector_mode': 'full_pool_greedy_mmr_adaptive_topk',
            'diversity_weight': DIVERSITY_WEIGHT,
            'adaptive_threshold': ADAPTIVE_SCORE_GAP_THRESHOLD,
            'context_top_k': CONTEXT_TOP_K,
            'policy_action': 'plackett_luce_mmr_without_replacement_random_k',
            'epochs': EPOCHS,
            'batch_size': BATCH_SIZE,
            'backbone_lr': BACKBONE_LR,
            'head_lr': HEAD_LR,
            'selector_lr': TACS_SELECTOR_LR,
            'candidate_pool_fraction': CANDIDATE_POOL_FRACTION,
            'candidate_pool_size': len(candidate_df),
            'tacs_use_full_candidate_pool': TACS_USE_FULL_CANDIDATE_POOL,
            'tacs_gumbel_tau': TACS_GUMBEL_TAU,
            'tacs_policy_weight': TACS_POLICY_WEIGHT,
            'tacs_entropy_weight': TACS_ENTROPY_WEIGHT,
            'supcon_weight': SUPCON_WEIGHT,
            'use_bbox_crop': USE_BBOX_CROP,
            'image_size': IMAGE_SIZE,
            'dino_model': DINOV3_MODEL_NAME,
            'use_context_label_prior': USE_CONTEXT_LABEL_PRIOR,
            'fusion_mode': getattr(model, 'active_fusion_mode', 'unknown'),
        },
    }


RESUME_FROM_LAST_CHECKPOINT = False
history = []
start_epoch = 1
best_val_loss = float('inf')
best_epoch = 0
bad_epochs = 0

if RESUME_FROM_LAST_CHECKPOINT and last_checkpoint_path.is_file():
    checkpoint = torch.load(last_checkpoint_path, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    scaler.load_state_dict(checkpoint['scaler_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    history = checkpoint.get('history', [])
    metrics = checkpoint.get('metrics', {})
    best_val_loss = metrics.get('best_val_loss', min((row.get('val_loss', float('inf')) for row in history), default=float('inf')))
    best_epoch = metrics.get('best_epoch', 0)
    print(f'Resuming from epoch {start_epoch}')

for epoch in range(start_epoch, EPOCHS + 1):
    backbone_trainable = epoch > FREEZE_BACKBONE_EPOCHS
    candidate_feature_bank = refresh_candidate_feature_bank(model, candidate_image_bank)
    train_metrics = train_one_epoch(model, train_loader, epoch, candidate_feature_bank, backbone_trainable=backbone_trainable)
    val_metrics = evaluate_one_epoch(model, val_loader, candidate_feature_bank, desc=f'Epoch {epoch:03d} [val full-pool]')
    scheduler.step()
    is_best = val_metrics['loss'] < best_val_loss - 1e-6
    if is_best:
        best_val_loss, best_epoch, bad_epochs = val_metrics['loss'], epoch, 0
    else:
        bad_epochs += 1
    lr_values = current_lrs(optimizer)
    record = {
        'epoch': epoch,
        'train_loss': train_metrics['loss'],
        'train_ce_loss': train_metrics.get('ce_loss'),
        'train_supcon_loss': train_metrics.get('supcon_loss'),
        'train_policy_loss': train_metrics.get('policy_loss'),
        'train_reward': train_metrics.get('reward'),
        'train_selector_entropy': train_metrics.get('selector_entropy'),
        'train_selected_same_class_any_context_rate': train_metrics.get('selected_same_class_rate'),
            'train_mean_context_count': train_metrics.get('mean_context_count'),
        'train_accuracy': train_metrics['accuracy'],
        'train_macro_f1': train_metrics['macro_f1'],
        'val_loss': val_metrics['loss'],
        'val_accuracy': val_metrics['accuracy'],
        'val_macro_f1': val_metrics['macro_f1'],
        'val_selected_same_class_any_context_rate': val_metrics.get('selected_same_class_rate'),
            'val_mean_context_count': val_metrics.get('mean_context_count'),
        'val_selector_entropy': val_metrics.get('selector_entropy'),
        'best_val_loss': best_val_loss,
        'best_epoch': best_epoch,
        'bad_epochs': bad_epochs,
        'backbone_trainable': backbone_trainable,
        'backbone_lr': lr_values.get('backbone'),
        'selector_lr': lr_values.get('selector'),
        'head_lr': lr_values.get('head'),
        'fusion_mode': getattr(model, 'active_fusion_mode', 'unknown'),
    }
    history.append(record)
    payload = build_checkpoint_payload(epoch, train_metrics, val_metrics, best_val_loss, best_epoch, history)
    torch.save(payload, last_checkpoint_path)
    if is_best:
        torch.save(payload, best_checkpoint_path)
    print(
        f"Epoch {epoch:03d}/{EPOCHS} | "
        f"train_loss={train_metrics['loss']:.4f} ce={train_metrics.get('ce_loss', 0):.4f} "
        f"policy={train_metrics.get('policy_loss', 0):.4f} reward={train_metrics.get('reward', 0):.4f} "
        f"train_acc={train_metrics['accuracy']:.4f} train_f1={train_metrics['macro_f1']:.4f} "
        f"sel_any={train_metrics.get('selected_same_class_rate', 0):.4f} meanK={train_metrics.get('mean_context_count', 0):.2f} | "
        f"val_loss={val_metrics['loss']:.4f} val_acc={val_metrics['accuracy']:.4f} "
        f"val_f1={val_metrics['macro_f1']:.4f} val_sel_any={val_metrics.get('selected_same_class_rate', 0):.4f} meanK={val_metrics.get('mean_context_count', 0):.2f} | "
        f"best_val={best_val_loss:.4f}@{best_epoch:03d} bad={bad_epochs}/{PATIENCE} | "
        f"bb_train={backbone_trainable} bb_lr={lr_values.get('backbone'):.2e} "
        f"sel_lr={lr_values.get('selector'):.2e} head_lr={lr_values.get('head'):.2e}"
    )
    if bad_epochs >= PATIENCE:
        print(f'Early stopping at epoch {epoch}; best validation loss was {best_val_loss:.4f} at epoch {best_epoch}.')
        break

final_train_metrics = train_metrics
final_val_metrics = val_metrics
final_payload = build_checkpoint_payload(epoch, final_train_metrics, final_val_metrics, best_val_loss, best_epoch, history)
torch.save(final_payload, checkpoint_path)
print('Saved final checkpoint to:', checkpoint_path.resolve())
if best_checkpoint_path.is_file():
    print('Saved best validation checkpoint to:', best_checkpoint_path.resolve())
history_df = pd.DataFrame(history)
history_df.to_csv(OUT_PUT_DIR / 'training_history.csv', index=False)
display(history_df.tail())


refresh full candidate feature bank:   0%|          | 0/28 [00:00<?, ?it/s]

/tmp/ipykernel_163/2411397457.py:397: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=DEVICE.type == 'cuda'):


Epoch 001 [adaptive diverse train]:   0%|          | 0/280 [00:00<?, ?it/s]

/tmp/ipykernel_163/118561078.py:144: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=DEVICE.type == 'cuda'):


Epoch 001 [val full-pool]:   0%|          | 0/32 [00:00<?, ?it/s]

/tmp/ipykernel_163/118561078.py:200: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=DEVICE.type == 'cuda'):


... (282 outputs left out)

/tmp/ipykernel_163/2411397457.py:397: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=DEVICE.type == 'cuda'):


Epoch 042 [adaptive diverse train]:   0%|          | 0/280 [00:00<?, ?it/s]

/tmp/ipykernel_163/118561078.py:144: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=DEVICE.type == 'cuda'):


KeyboardInterrupt: 

## 6. Vẽ training history

Dùng để kiểm tra loss/accuracy trong quá trình train. Benchmark chính nằm ở cell `Official test accuracy`.


In [9]:
import pandas as pd
if 'history_df' not in globals():
    history_df = pd.read_csv(OUT_PUT_DIR / 'training_history.csv')
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
history_df.plot(x='epoch', y='train_loss', ax=axes[0], marker='o', markevery=max(1, len(history_df) // 10), title='Training loss')
history_df.plot(x='epoch', y='train_accuracy', ax=axes[1], marker='o', markevery=max(1, len(history_df) // 10), title='Training top-1 accuracy')
for axis in axes:
    axis.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
display(history_df.tail())


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/outputs/tacs_adaptive_diverse_sun_true_val_split/training_history.csv'

## 7. Validation threshold va official test

Cell nay chon adaptive threshold dua tren validation loss, sau do danh gia mot lan tren official test va luu so context thuc te cho tung query.


## 8. Inference

Ham demo dung MMR va threshold da chon tren validation de tra ve 1-3 context cho query.


In [10]:



import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

def macro_f1_score_np(y_true, y_pred, num_classes):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    f1_values = []
    for class_id in range(num_classes):
        tp = np.sum((y_true == class_id) & (y_pred == class_id))
        fp = np.sum((y_true != class_id) & (y_pred == class_id))
        fn = np.sum((y_true == class_id) & (y_pred != class_id))
        denom = 2 * tp + fp + fn
        f1_values.append(0.0 if denom == 0 else float((2 * tp) / denom))
    return float(np.mean(f1_values))

@torch.no_grad()
def evaluate_full_pool_context_model(network, loader, candidate_feature_bank, top_k=TOP_K,
                                     adaptive_threshold=ADAPTIVE_SCORE_GAP_THRESHOLD, desc='eval'):
    network.eval()
    bank=candidate_feature_bank.to(DEVICE,non_blocking=True)
    total_loss, targets, preds, rows=0.0,[],[],[]
    top_k=min(top_k,NUM_CLASSES)
    def get_class_name(cid):
        return class_names[cid] if 0 <= cid < len(class_names) else f"class_{cid}"
    for batch in tqdm(loader,desc=desc,leave=False):
        moved=move_query_batch_to_device(batch)
        with torch.amp.autocast('cuda',enabled=DEVICE.type=='cuda'):
            details=network.select_full_pool(moved['query_images'],bank,use_gumbel=False)
            mask=adaptive_context_mask(details['selected_mmr_scores'],adaptive_threshold)
            context_images=gather_bank_images(details['selected_indices'])
            logits=network.forward_selected_context(
                moved['query_images'],context_images,context_mask=mask,return_features=False
            )
            loss=criterion(logits,moved['labels'])
        probs=torch.softmax(logits.float(),dim=1).cpu()
        values,indices=torch.topk(probs,k=top_k,dim=1)
        pred,labels=logits.argmax(1).cpu(),moved['labels'].cpu()
        positions=details['selected_indices'].detach().cpu()
        mmr=details['selected_mmr_scores'].detach().cpu()
        raw=details['selector_scores'].detach().cpu()
        masks=mask.detach().cpu()
        total_loss+=loss.item()*len(labels); targets.extend(labels.tolist()); preds.extend(pred.tolist())
        for ri,(fname,label,pred_i,value_row,index_row) in enumerate(zip(batch['fname'],labels,pred,values,indices)):
            true_id,pred_id=int(label),int(pred_i)
            ranks=torch.where(masks[ri])[0].tolist()
            pool_pos=[int(positions[ri,r]) for r in ranks]
            ctx=[candidate_df.iloc[pos] for pos in pool_pos]
            ctx_ids=[int(row['class_id']) for row in ctx]
            ctx_files=[str(row['fname']) for row in ctx]
            ctx_names=[get_class_name(cid) for cid in ctx_ids]
            ctx_mmr=[float(mmr[ri,r]) for r in ranks]
            same=[cid==true_id for cid in ctx_ids]
            rows.append({
                'fname':fname,'true_class_id':true_id,'true_class_name':get_class_name(true_id),
                'context_count':len(pool_pos),'context_fnames':'|'.join(ctx_files),
                'context_class_ids':'|'.join(map(str,ctx_ids)),'context_class_names':'|'.join(ctx_names),
                'context_pool_positions':'|'.join(map(str,pool_pos)),
                'context_mmr_scores':'|'.join(f'{x:.8f}' for x in ctx_mmr),
                'context_selector_scores':'|'.join(f'{float(raw[ri,pos]):.8f}' for pos in pool_pos),
                'top1_context_fname':ctx_files[0],'top1_context_class_id':ctx_ids[0],
                'top1_context_class_name':ctx_names[0],'top1_context_pool_position':pool_pos[0],
                'top1_mmr_score':ctx_mmr[0],'same_class_context_top1':same[0],
                'same_class_context_any':any(same),'same_class_context_fraction':float(np.mean(same)),
                'predicted_class_id':pred_id,'predicted_class_name':get_class_name(pred_id),
                'correct':pred_id==true_id,'confidence':float(value_row[0]),
                'top_k_class_ids':'|'.join(str(int(x)) for x in index_row),
            })
    acc=float(np.mean([r['correct'] for r in rows])) if rows else 0.0
    counts=pd.Series([r['context_count'] for r in rows]).value_counts().sort_index().to_dict()
    return {'loss':total_loss/max(len(loader.dataset),1),'accuracy':acc,
            'macro_f1':macro_f1_score_np(targets,preds,NUM_CLASSES),
            'mean_context_count':float(np.mean([r['context_count'] for r in rows])) if rows else 0.0,
            'context_count_distribution':counts,
            'test_context_top1_same_class_rate':float(np.mean([r['same_class_context_top1'] for r in rows])) if rows else 0.0,
            'test_context_any_same_class_rate':float(np.mean([r['same_class_context_any'] for r in rows])) if rows else 0.0,
            'predictions':pd.DataFrame(rows)}

# 1. Nạp checkpoint tốt nhất để đánh giá
eval_checkpoint_path = best_checkpoint_path if best_checkpoint_path.is_file() else checkpoint_path
print('Loaded checkpoint for evaluation:', eval_checkpoint_path.resolve())
checkpoint = torch.load(eval_checkpoint_path, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(DEVICE)
model.eval()

# 2. Đảm bảo danh sách class_names đủ 397 classes
if 'class_names' not in globals() or len(class_names) != NUM_CLASSES:
    class_names = load_class_names(DATASET_ROOT)

# 3. Làm mới candidate feature bank bằng backbone đã huấn luyện
candidate_feature_bank = refresh_candidate_feature_bank(model, candidate_image_bank)
torch.save({
    'candidate_feature_bank': candidate_feature_bank.cpu(),
    'candidate_labels': candidate_labels_tensor.cpu(),
    'candidate_fnames': candidate_fnames,
}, OUT_PUT_DIR / 'trained_tacs_full_pool_candidate_feature_bank.pt')

# Chọn ngưỡng adaptive chỉ trên validation; test chỉ được chấm sau khi khóa ngưỡng.
threshold_rows=[]
for threshold in ADAPTIVE_THRESHOLD_GRID:
    vm=evaluate_full_pool_context_model(
        model,val_loader,candidate_feature_bank,top_k=TOP_K,
        adaptive_threshold=threshold,desc=f'val threshold={threshold:g}'
    )
    threshold_rows.append({'threshold':float(threshold),'loss':vm['loss'],'accuracy':vm['accuracy'],
                           'macro_f1':vm['macro_f1'],'mean_context_count':vm['mean_context_count']})
threshold_results_df=pd.DataFrame(threshold_rows).sort_values(
    ['loss','accuracy'],ascending=[True,False],kind='stable'
).reset_index(drop=True)
BEST_ADAPTIVE_THRESHOLD=float(threshold_results_df.iloc[0]['threshold'])
threshold_results_df.to_csv(OUT_PUT_DIR/'adaptive_threshold_validation.csv',index=False)
print('Validation-selected adaptive threshold:',BEST_ADAPTIVE_THRESHOLD)
display(threshold_results_df)

results=evaluate_full_pool_context_model(
    model,test_loader,candidate_feature_bank,top_k=TOP_K,
    adaptive_threshold=BEST_ADAPTIVE_THRESHOLD,desc='test adaptive diverse contexts'
)
test_prediction_df=results.pop('predictions')
test_prediction_df.to_csv(OUT_PUT_DIR/'test_predictions.csv',index=False)
metrics=results
metrics['adaptive_threshold']=BEST_ADAPTIVE_THRESHOLD
metrics['diversity_weight']=DIVERSITY_WEIGHT
print(f"test_loss: {metrics['loss']:.4f}")
print(f"test_accuracy: {metrics['accuracy']:.4f}")
print(f"test_macro_f1: {metrics['macro_f1']:.4f}")
print(f"mean_context_count: {metrics['mean_context_count']:.3f}")
print(f"context_count_distribution: {metrics['context_count_distribution']}")
print(f"top1_same_class_rate: {metrics['test_context_top1_same_class_rate']:.4f}")
print(f"any_same_class_rate: {metrics['test_context_any_same_class_rate']:.4f}")
display(test_prediction_df.head())

Loaded checkpoint for evaluation: /kaggle/working/outputs/tacs_adaptive_diverse_sun_true_val_split/best_val_checkpoint.pt


refresh full candidate feature bank:   0%|          | 0/28 [00:00<?, ?it/s]

/tmp/ipykernel_163/2411397457.py:397: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=DEVICE.type == 'cuda'):


val threshold=0:   0%|          | 0/32 [00:00<?, ?it/s]

val threshold=0.25:   0%|          | 0/32 [00:00<?, ?it/s]

val threshold=0.5:   0%|          | 0/32 [00:00<?, ?it/s]

... (5 outputs left out)

,threshold,loss,accuracy,macro_f1,mean_context_count
0,1.50,1.715541,0.777330,0.775875,2.663980
1,2.00,1.715696,0.777330,0.775842,2.840302
2,1.00,1.716432,0.775819,0.774209,2.289169
3,0.75,1.717247,0.778338,0.776464,1.991940
4,0.00,1.717529,0.780856,0.778780,1.000000
5,0.50,1.718179,0.779849,0.777971,1.574811
6,0.25,1.718271,0.780353,0.778348,1.123929


test adaptive diverse contexts:   0%|          | 0/311 [00:00<?, ?it/s]

test_loss: 1.7810
test_accuracy: 0.7588
test_macro_f1: 0.7580
mean_context_count: 2.671
context_count_distribution: {1: 1998, 2: 2530, 3: 15322}
top1_same_class_rate: 0.6222
any_same_class_rate: 0.7815


,fname,true_class_id,true_class_name,context_count,context_fnames,context_class_ids,context_class_names,context_pool_positions,context_mmr_scores,context_selector_scores,...,top1_context_pool_position,top1_mmr_score,same_class_context_top1,same_class_context_any,same_class_context_fraction,predicted_class_id,predicted_class_name,correct,confidence,top_k_class_ids
0,sun_ajkqrqitspwywirx.jpg,0,a/abbey,2,sun_aakbdcgfpksytcwj.jpg|sun_alqgvwyohunprcsh.jpg,0|0,a/abbey|a/abbey,0|2,13.76459026|12.27043915,0.89892578|0.82128906,...,0,13.764590,True,True,1.0,85,c/cathedral/outdoor,False,0.391972,85|0|39|73|95
1,sun_ajhtswxgrqbeiikc.jpg,0,a/abbey,3,sun_acyyxelzhddpmokt.jpg|sun_aaemixhfquhnxxlm....,209|375|389,k/kasbah|v/village|w/wheat_field,1904|3378|3514,5.89916897|5.74636793|5.73133183,0.36401367|0.36181641|0.35546875,...,1904,5.899169,False,False,0.0,237,m/mausoleum,False,0.119395,237|209|0|375|358
2,sun_ahebfjooupcqbjht.jpg,0,a/abbey,3,sun_bjlukomljlmikwdy.jpg|sun_aomfcoftnyqdnvle....,239|82|357,m/moat/water|c/castle|t/topiary_garden,2141|766|3220,6.13887453|5.44979477|4.80947304,0.41284180|0.37719727|0.32617188,...,2141,6.138875,False,False,0.0,0,a/abbey,True,0.247969,0|231|263|115|82
3,sun_amdwiijwnagslgef.jpg,0,a/abbey,3,sun_bxflqprgluymhlhh.jpg|sun_anjhvwnbagbgtkgu....,9|84|9,a/apse/indoor|c/cathedral/indoor|a/apse/indoor,80|777|72,8.86748505|8.22796154|7.58150482,0.59130859|0.56201172|0.52148438,...,80,8.867485,False,False,0.0,84,c/cathedral/indoor,False,0.933808,84|9|343|94|241
4,sun_awzqxzqukmbujwzm.jpg,0,a/abbey,2,sun_aakbdcgfpksytcwj.jpg|sun_budrypozqglzxwci.jpg,0|85,a/abbey|c/cathedral/outdoor,0|779,10.20510769|9.30694866,0.61230469|0.57275391,...,0,10.205108,True,True,0.5,0,a/abbey,True,0.605544,0|344|64|85|263


In [11]:
@torch.no_grad()
def find_tacs_contexts_for_image(image_path, top_k=CONTEXT_TOP_K):
    image_path=Path(image_path)
    image=eval_transform(Image.open(image_path).convert('RGB')).unsqueeze(0).to(DEVICE)
    network=globals().get('model')
    network.eval()
    if 'candidate_feature_bank' not in globals():
        globals()['candidate_feature_bank']=refresh_candidate_feature_bank(network,candidate_image_bank)
    with torch.amp.autocast('cuda',enabled=DEVICE.type=='cuda'):
        details=network.select_full_pool(image,candidate_feature_bank.to(DEVICE),use_gumbel=False)
    threshold=globals().get('BEST_ADAPTIVE_THRESHOLD',ADAPTIVE_SCORE_GAP_THRESHOLD)
    mask=adaptive_context_mask(details['selected_mmr_scores'],threshold)[0]
    ranks=torch.where(mask)[0].tolist()[:min(int(top_k),CONTEXT_TOP_K)]
    positions=details['selected_indices'][0].detach().cpu()
    rows=[candidate_df.iloc[int(positions[r])] for r in ranks]
    scores=[float(details['selected_mmr_scores'][0,r]) for r in ranks]
    return rows,scores

def load_tacs_model_for_inference(checkpoint_file, device=DEVICE):
    saved = torch.load(checkpoint_file, map_location=device)
    loaded_model = DinoV3TACSClassifier(
        repo_dir=DINOV3_REPO_DIR,
        model_name=DINOV3_MODEL_NAME,
        weights_path=resolve_dinov3_weights(),
        num_classes=len(saved['class_names']),
        feature_dim=FEATURE_DIM,
        allow_feature_fallback=False,
    )
    loaded_model.load_state_dict(saved['model_state_dict'])
    return loaded_model.to(device).eval(), saved['class_names']


model, class_names = load_tacs_model_for_inference(best_checkpoint_path)


def predict_one_image_with_tacs_context(image_path, context_image_paths=None, network=model, top_k=TOP_K):
    if network is None:
        network = model
    image_path = Path(image_path)
    if context_image_paths is None:
        context_rows, tacs_scores = find_tacs_contexts_for_image(image_path, top_k=CONTEXT_TOP_K)
        context_image_paths = [Path(row['image_path']) for row in context_rows]
        context_names = [class_names[int(row['class_id'])] for row in context_rows]
        context_labels = [int(row['class_id']) for row in context_rows]
    else:
        if isinstance(context_image_paths, (str, Path)):
            context_image_paths = [context_image_paths]
        context_image_paths = [Path(path) for path in context_image_paths]
        context_names = [None] * len(context_image_paths)
        context_labels = [0] * len(context_image_paths)
        tacs_scores = [0.0] * len(context_image_paths)

    query_image = eval_transform(Image.open(image_path).convert('RGB')).unsqueeze(0).to(DEVICE)
    context_images = torch.stack([eval_transform(Image.open(path).convert('RGB')) for path in context_image_paths], dim=0).unsqueeze(0).to(DEVICE)
    context_scores = torch.tensor([tacs_scores], dtype=torch.float32, device=DEVICE)
    context_labels_tensor = torch.tensor([context_labels], dtype=torch.long, device=DEVICE)

    network.eval()
    with torch.inference_mode():
        context_mask = torch.ones((1, context_images.size(1)), dtype=torch.bool, device=DEVICE)
        logits = network.forward_selected_context(query_image, context_images, context_mask=context_mask, return_features=False)
        probs = torch.softmax(logits, dim=1)[0]
        values, indices = torch.topk(probs, k=min(top_k, NUM_CLASSES))

    print('Query:', image_path)
    for rank, (context_path, context_name, tacs_score) in enumerate(zip(context_image_paths, context_names, tacs_scores), start=1):
        print(f'TACS context {rank}:', context_path)
        if context_name is not None:
            print(f'  label: {context_name}')
        print(f'  TACS utility score: {tacs_score:.4f}')
    print('\nTop predictions:')
    for rank, (class_id, probability) in enumerate(zip(indices.tolist(), values.tolist()), start=1):
        print(f'{rank}. {class_names[class_id]} | p={probability:.4f}')


print('Final checkpoint:', checkpoint_path.resolve())
print('Best-val checkpoint:', best_checkpoint_path.resolve())
print('Latest checkpoint:', last_checkpoint_path.resolve())
print('Method:', 'TACS (Ours) - full candidate pool shared-backbone selector')


Final checkpoint: /kaggle/working/outputs/tacs_adaptive_diverse_sun_true_val_split/final_tacs_full_pool_dinov3_vits16_sun397.pt
Best-val checkpoint: /kaggle/working/outputs/tacs_adaptive_diverse_sun_true_val_split/best_val_checkpoint.pt
Latest checkpoint: /kaggle/working/outputs/tacs_adaptive_diverse_sun_true_val_split/checkpoint_last.pt
Method: TACS (Ours) - full candidate pool shared-backbone selector


In [12]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

## 9. Ghi chu benchmark

- Bien the TACS Adaptive Top-K + MMR diversity, khong phai TACS nguyen ban.
- Selector cham diem full candidate pool 20%; MMR chon toi da 3 anh co utility cao va it trung lap.
- Train random K=1..3 theo batch; Gumbel-ST va policy reward dung diversity-aware scores.
- Threshold duoc chon tren validation; khong dung test de chon K.
- test_predictions.csv ghi K va filename/class/score context; metrics co phan bo K=1/2/3.
- Giu split, candidate pool, DINOv3, augmentation va optimizer/loss; batch size 2.
- So sanh voi baseline tren cung split/seed/candidate pool va bao cao nhieu seed.


In [ ]:

# --- CELL CUỐI: TỰ ĐỘNG DỪNG TÍNH TOÁN & SHUTDOWN PHIÊN ---
import os
import time
import torch

print("=== QUÁ TRÌNH HUẤN LUYỆN VÀ ĐÁNH GIÁ ĐÃ HOÀN TẤT ===")

# 1. Giải phóng toàn bộ VRAM
if torch.cuda.is_available():
    torch.cuda.empty_cache()

time.sleep(2)

# 2. Tắt Kernel để dừng ngay lập tức mọi tác vụ đang chạy
try:
    from ipykernel.kernelapp import IPKernelApp
    IPKernelApp.instance().shutdown()
except Exception:
    os._exit(0)

=== QUÁ TRÌNH HUẤN LUYỆN VÀ ĐÁNH GIÁ ĐÃ HOÀN TẤT ===


In [ ]:
# import numpy as np
# import pandas as pd
# from PIL import Image
# from tqdm import tqdm

# def analyze_dataset_dimensions(df, check_bbox=True):
#     widths, heights = [], []
#     crop_w_list, crop_h_list = [], []

#     for _, row in tqdm(df.iterrows(), total=len(df), desc="Đang quét kích thước ảnh"):
#         # Image.open chỉ đọc metadata header, cực nhanh và không tốn RAM
#         with Image.open(row['image_path']) as img:
#             w, h = img.size
#             widths.append(w)
#             heights.append(h)

#         if check_bbox and 'bbox_x1' in row:
#             left = max(0, int(row['bbox_x1']) - 1)
#             top = max(0, int(row['bbox_y1']) - 1)
#             right = max(left + 1, int(row['bbox_x2']))
#             bottom = max(top + 1, int(row['bbox_y2']))
#             crop_w_list.append(right - left)
#             crop_h_list.append(bottom - top)

#     data = {
#         "Metric": ["Min", "Max", "Mean", "Median"],
#         "Orig Width (W)": [np.min(widths), np.max(widths), round(np.mean(widths), 1), np.median(widths)],
#         "Orig Height (H)": [np.min(heights), np.max(heights), round(np.mean(heights), 1), np.median(heights)],
#     }

#     if check_bbox and crop_w_list:
#         data["Crop Width (BBox)"] = [np.min(crop_w_list), np.max(crop_w_list), round(np.mean(crop_w_list), 1), np.median(crop_w_list)]
#         data["Crop Height (BBox)"] = [np.min(crop_h_list), np.max(crop_h_list), round(np.mean(crop_h_list), 1), np.median(crop_h_list)]

#     stats_df = pd.DataFrame(data)
    
#     # Tính tỉ lệ khung hình (Aspect Ratio W/H)
#     aspect_ratios = np.array(widths) / np.array(heights)
#     print(f"Aspect ratio gốc trung bình (W/H): {aspect_ratios.mean():.2f}")
#     print(f"Số ảnh có W > H (ảnh ngang): {np.sum(aspect_ratios > 1.0)} / {len(df)} ({np.mean(aspect_ratios > 1.0):.1%})")
    
#     return stats_df

# # Kiểm tra trên tập train
# train_size_stats = analyze_dataset_dimensions(full_train_df, check_bbox=USE_BBOX_CROP)
# display(train_size_stats)

In [ ]:
# --- CELL DỪNG NOTEBOOK & GIẢI PHÓNG TÀI NGUYÊN ---
import gc
import os
import time
import torch

print("=== HOÀN TẤT: ĐANG DỌN TÀI NGUYÊN VÀ DỪNG PHIÊN ===")

# 1. Thu hồi bộ nhớ RAM và VRAM
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

time.sleep(2)

# 2. Dừng và đóng Kernel ngay lập tức
try:
    from ipykernel.kernelapp import IPKernelApp
    IPKernelApp.instance().shutdown()
except Exception:
    os._exit(0)